In [36]:
import torch
import torch.nn as nn
from torch.utils import data
import torchvision
from torchvision import transforms
from tqdm.notebook import tqdm
import time
import os

import dataLoader

# Get GPU if possible, else get CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("Using the GPU!")
else:
    print("WARNING: Could not find GPU! Using CPU only")

Using the GPU!


### Relevant Paths

In [15]:
data_dir = '/workspace/data/cv_project'

train_data = data_dir + '/grid_data_train.h5'
val_data = data_dir + '/grid_data_val.h5'
test_data = data_dir + 'grid_data_val.h5'

### Data Preparation Functions

Some functions to create the transforms for the dataloaders and then instantiate them.

In [25]:
def get_image_transforms():
    
    """
    Make dictionary of transforms for sets used for training/evaluation
    
    Returns
    -------
    transforms: dict, transforms for training/evaluation datasets
    """
    
    transform_eval = transforms.Compose([
        transforms.Normalize([0.6303], [0.1225])
    ])
    
    transform_train = transforms.Compose([
        transforms.Normalize([0.6303], [0.1225])
    ])
    
    transform_dict = {'train': transform_train, 'eval': transform_eval}
    
    return transform_dict
    

def get_dataloaders(data_dir, batch_size, max_range, shuffle=True, transform=None):
    
    """
    Make dictionary of dataloaders for the train, validation, and test sets.
    
    Parameters
    ----------
    data_dir: str, directory that contains the data .h5 files
    batch_size: int, numbere of examples per batch
    shuffle: bool, whether or not to shuffle the training set
    transform: dict, dictionary of transforms with keys 'train' and 'eval'
    
    Returns
    -------
    dataloaders: dict, dictionary with data loaders for each dataset
    """
    
    # Transform dictionary
    data_transforms = {
        'train': transform['train'] if transform is not None else transform,
        'val': transform['eval'] if transform is not None else transform,
        'test': transform['eval'] if transform is not None else transform
    }
    
    # Create dataset
    datasets = {x: dataLoader.Gunshot(dir_path=os.path.join(data_dir, 'grid_data_{}.h5'.format(x)), max_range=max_range, transform=data_transforms[x]) for x in data_transforms.keys()}
    
    # Make dataloaders
    dataloaders = {x: data.DataLoader(datasets[x], batch_size=batch_size, shuffle=False if x != 'train' else shuffle) for x in data_transforms.keys()}
    
    return dataloaders

### Training

Helper function to train/validate the model

In [35]:
def train(model, dataloaders, criterion, optimizer, num_epochs, save_dir=None, save_all_epochs=False):
    
    """
    Helper function with some features to train the model and save the most
    recent epoch with the lowest validation MSE.
    
    Parameters
    ----------
    model: nn.Module, model object.
    dataloaders: dict, dictionary containing at least the keys
                 'train', 'val', and 'test' which map to dataloaders
                 for these datasets.
    criterion: torch.nn.Loss, Loss function.
    optimizer: torch.optim, method of weight updating.
    num_epochs: int, number of epochs to train the model.
    save_dir: str, directory where models will be saved to. Make
              None to not write anything to disk.
    save_all_epochs: bool, whether to save the model weights for all
                     epochs, or just the best MSE weights.
    
    Returns
    -------
    model: nn.Module, model object with weights for best validation set MSE.
    val_mse_history: list, validation set MSE over all epochs
    train_mse_history: list, train set MSE over all epochs
    """
    
    # Time training
    since = time.time()
    
    # History vectors
    val_mse_history = []
    train_mse_history = []
    
    # Initialize best model
    best_model_wts = copy.deepcopy(mode.state_dict())
    best_mse = 0.0
    
    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch + 1, num_epochs))
        print('-'*10)
        
        # Train + evaluate on validation data each epoch
        for phase in ['train', 'val']:
            if phase == 'train':
                # Put model in train mode
                model.train()
            else:
                # Put model in evaluation mode
                model.eval()
            
            # Keep track of loss and raw error
            running_loss = 0.0
            running_sq_error = 0.0
            
            # Flip through data batches
            for inputs, labels in tqdm(dataloaders[phase]):
                
                # Put data/labels on device
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                # Zero out gradient for new batch
                optimizer.zero_grad()
                
                # Update weights
                with torch.set_grad_enabled(phase == 'train'):
                    
                    # Get model outputs and loss
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    
                    # Backprop if we are training
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                # Update running statistics
                running_loss +=  loss.item() * inputs.size(0)
                running_sq_error += ((labels - outputs) ** 2).sum().item()
                
            # Stats to display
            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_mse = running_sq_error / len(dataloaders[phase].dataset)
            
            # Print epoch info
            print("{} Loss: {:.4f} MSE: {:.4f}".format(phase, epoch_loss, epoch_mse))
            
            # Deep copy model
            if phase == 'val' and epoch_mse < best_mse:
                best_mse = epoch_mse
                best_model_wts = copy.deepcopy(model.state_dict())
            if phase == 'train':
                train_mse_history.append(epoch_mse)
            if phase == 'val':
                val_mse_history.append(epoch_mse)
            if phase == 'train' and save_all_epochs:
                torch.save(model.state_dict(), os.path.join(save_dir, 'weights_{}.pt'.format(epoch)))
            
        print()
        
    # Training done!
    time_elapsed = time.time() - since
    print('Training completed in {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))
    print('Best val MSE: {:.4f}'.format(best_mse))
    
    # Save best model weights and load them into the model before returning
    torch.save(best_model_wts, os.path.join(save_dir, 'weights_best_val_mse.pt'))
    if not save_all_epochs:
        torch.save(model.state_dict(), os.path.join(save_dir, 'weights_last.pt'.format(epoch)))
    mode.load_state_dict(best_model_wts)
    
    return model, val_mse_history, train_mse_history

Set up optimizer and loss.

In [37]:
def make_optimizer(model, learning_rate):
    
    """
    Create opimizer to train model.
    
    Returns
    -------
    optimizer: torch.optim, optimizer object.
    """
    
    # Make optimizer object
    optimizer = torch.optim.Adam(mode.parameters(), lr=learning_rate)
    return optimizer

def get_loss():
    
    """
    Create loss object
    
    Returns
    -------
    criterion: torch.nn, loss object.
    """
    
    # Make loss object
    criterion = nn.MSELoss()
    return criterion